In [1]:
from data_process import *
from mcar.simulate import *
from graphOUedge import GraphOUEdge

def pred_robust(seed: int, p: float, gamma: float, num_node: int, scale: float, T: int, dir: str, forecast_step: int):
    TICKERS = [str(i) for i in range (num_node)]
    G = nx.erdos_renyi_graph(num_node, p, seed=seed)
    G = nx.relabel_nodes(G, dict(enumerate(TICKERS)))
    A = nx.to_numpy_array(G)

    adj_sim = adj_to_line_graph(A, TICKERS)
    adj_L_sim = adjacency_matrices(adj_sim[0], 1)
    W_C_sim = edge_weight(adj_L_sim)
    X_sim = GraphOUEdge([1], len(adj_sim[1]), 1, adj_L_sim, W_C_sim, gamma, dir, scale=scale, T=T)
    X_sim.alpha_set(np.array([[5]+[1 for _ in range(len(adj_sim[1]) - 1)]]))
    X_sim.beta_set([[2]])
    X_sim.Q_matrix()
    data_sim = X_sim.simulate('CP', 2, seed=1000, rate=1)

    edges = list(G.edges())
    G_without_first = G.copy()
    first_edge = edges[0]
    G_without_first.remove_edge(*first_edge)
    A_without_first = nx.to_numpy_array(G_without_first, nodelist=TICKERS)

    G_without_last = G.copy()
    last_edge = edges[-1]
    G_without_last.remove_edge(*last_edge)
    A_without_last = nx.to_numpy_array(G_without_last, nodelist=TICKERS)

    model_pred(data=data_sim, test_size=400, adj_sim=adj_sim, forecast_step=forecast_step, P_Q_ratio=T**4, gamma=gamma, label='correct', scenario=dir)
    model_pred(data=data_sim, test_size=400, adj_sim=adj_to_line_graph(A_without_first, TICKERS), forecast_step=forecast_step, P_Q_ratio=T**4, gamma=gamma, label='missing_5', scenario=dir)
    model_pred(data=data_sim, test_size=400, adj_sim=adj_to_line_graph(A_without_last, TICKERS), forecast_step=forecast_step, P_Q_ratio=T**4, gamma=gamma, label='missing_1', scenario=dir)

    boxplot_(data_sim, dir)

In [10]:
pred_robust(123, 0.15, 1.01, 20, 1, 3, 'test', 1)

AssertionError: 